# DS2002 · JSON to Tidy Tables

**Studio — 2026-09-30 · Fall 2026**  
**Class time:** 45 minutes

---

## One response, straight off the wire

Monday you took apart a payload that had already been laid out for you, indented and readable. Real responses do not arrive that way. Whitespace costs bandwidth, so APIs ship their JSON with every space squeezed out of it.

Here is a portfolio response from a market-data service, exactly as it came down. Run the cell and try to read it.

In [1]:
RAW = '{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple Inc.","sector":"Technology"},"shares":120,"prices":[{"date":"2026-09-28","close":231.4,"volume":52300000},{"date":"2026-09-29","close":229.85,"volume":48100000},{"date":"2026-09-30","close":233.1,"volume":55600000}]},{"ticker":"MSFT","company":{"name":"Microsoft Corp.","sector":"Technology"},"shares":45,"prices":[{"date":"2026-09-28","close":502.1,"volume":22100000},{"date":"2026-09-29","close":"498.75","volume":0},{"date":"2026-09-30","close":505.3,"volume":24800000}]},{"ticker":"KO","company":{"name":"Coca-Cola Co."},"shares":300,"prices":[{"date":"2026-09-28","close":71.25,"volume":12400000},{"date":"2026-09-29","close":null,"volume":13000000},{"date":"2026-09-30","close":70.9,"volume":11800000}]}]}}'
print('characters:', len(RAW))
print(RAW[:160], '...')

characters: 871
{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple I ...


Eight hundred and seventy-one characters on one line. You cannot read that, and neither can anyone else. That is fine. **Nobody reads raw JSON.** You parse it, then you ask Python to lay it out for you.

Notice what you are holding right now: `RAW` is a *string*. It has a length. You could slice it. There are no keys to look up yet, because text does not have keys.

### Step 1 — parse it, then make it readable

Two different jobs, and people confuse them constantly. `json.loads` turns text into Python objects. `json.dumps` with `indent` turns Python objects back into text that a human can scan. You are about to do both, in that order, and end up looking at the same data you started with — just formatted.

In [2]:
import json, pandas as pd

data = json.loads(RAW)
print(type(RAW), '->', type(data))
print(json.dumps(data, indent=2)[:700], '...') #pushes it out

<class 'str'> -> <class 'dict'>
{
  "as_of": "2026-09-30",
  "currency": "USD",
  "source": "teaching-fixture",
  "portfolio": {
    "name": "Cavalier Growth",
    "holdings": [
      {
        "ticker": "AAPL",
        "company": {
          "name": "Apple Inc.",
          "sector": "Technology"
        },
        "shares": 120,
        "prices": [
          {
            "date": "2026-09-28",
            "close": 231.4,
            "volume": 52300000
          },
          {
            "date": "2026-09-29",
            "close": 229.85,
            "volume": 48100000
          },
          {
            "date": "2026-09-30",
            "close": 233.1,
            "volume": 55600000
          }
        ]
      },
      { ...


Same information, 871 characters versus something much longer, and now you can see the shape. `indent=2` is the first thing to reach for when a payload lands in your lap.

**TODO:** in one sentence, why is it wrong to say the API "sent you a dictionary"?

### Step 2 — map the shape before you touch it

Never normalize a payload you have not walked. You need two numbers before you go any further: how many holdings there are, and how many price rows you should end up with.

The first line is done for you — `.keys()` gives you an object's names when you do not already know them. The counting is yours.

**TODO:** print the portfolio name, the number of holdings, and the total number of price records across all holdings.

In [3]:
print('top-level keys:', list(data.keys()))

# TODO: the portfolio name
# TODO: how many holdings?
# TODO: how many price records in total? (len() on each holding's prices)
portfolio_name = data['portfolio']['name']
print('portfolio name:', portfolio_name)

holdings = data['portfolio']['holdings']
holdings_count = len(holdings)
print('number of holdings:', holdings_count)

total_price_records = sum(len(holding['prices']) for holding in holdings)
print('total price records:', total_price_records)

top-level keys: ['as_of', 'currency', 'source', 'portfolio']
portfolio name: Cavalier Growth
number of holdings: 3
total price records: 9


**TODO:** write your two numbers down here, because every assertion below checks against them.

- holdings: 3
- price rows: 9
- one row of the table you are about to build is one holding price record on a specific date

### Step 3 — trace a path, and hit the wall

A path is the sequence of names and positions that gets you to one value. Reading a path is the skill; the brackets are just typing.

**TODO:** print Apple's sector, then try to print Coca-Cola's the same way. One of those raises an exception. Catch it and read what it says. Then rewrite the line so it checks whether the name is there first and prints `'unknown'` when it is not — the `'pm25' not in r` test from Monday's missing/null/empty/zero table, pointed at `'sector'`.

In [4]:
# TODO: data['portfolio']['holdings'][0]['company']['sector']
# TODO: now try the same thing for KO -- what happens, and why?
# TODO: rewrite it using an `in` check so a missing sector prints 'unknown'


aapl_sector = data['portfolio']['holdings'][0]['company']['sector']
print("Apple's sector:", aapl_sector)

try:
    ko_sector = data['portfolio']['holdings'][2]['company']['sector']
    print("Coca-Cola's sector:", ko_sector)
except KeyError as e:
    print(f"Caught KeyError: key {e} is missing in Coca-Cola's company dictionary!")

ko_company = data['portfolio']['holdings'][2]['company']
ko_sector_safe = ko_company['sector'] if 'sector' in ko_company else 'unknown'
print("Coca-Cola's sector (safe check):", ko_sector_safe)

Apple's sector: Technology
Caught KeyError: key 'sector' is missing in Coca-Cola's company dictionary!
Coca-Cola's sector (safe check): unknown


### Step 4 — the four checks

It decoded and it parsed, so checks 1 and 2 passed the moment `json.loads` returned. Check 3 asks whether the data matches what you were promised, and check 4 asks whether it means what you think.

This payload has **four** planted problems. Three of them are in the price records and one is in a company object. None of them will raise an error.

**TODO:** walk every price record and report, with the ticker and date for each:

1. a `close` that came through as `None`
2. a `close` that is a *string* when every other one is a number
3. a `volume` of exactly `0`
4. a holding whose company object is missing a field the others have

Then answer the one that matters: **which of those four are genuinely missing data, and which are real measurements?** They do not get handled the same way.

In [5]:
# TODO: loop over holdings, then over prices, and report the four problems
#
# Hint for #2: isinstance(value, str)
# Hint for #3: write it as `volume == 0`, not `if not volume` -- ask yourself why
# Loop over holdings and price records to inspect problems
for holding in data['portfolio']['holdings']:
    ticker = holding['ticker']
    company = holding['company']

    # 4. Check for missing company fields
    if 'sector' not in company:
        print(f"Problem 4 (Missing Field): {ticker} company dictionary is missing 'sector'")

    for p in holding['prices']:
        date = p['date']
        close = p['close']
        volume = p['volume']

        # 1. Close is None (null)
        if close is None:
            print(f"Problem 1 (None close): {ticker} on {date}")

        # 2. Close is a string
        if isinstance(close, str):
            print(f"Problem 2 (String close): {ticker} on {date} (value: '{close}')")

        # 3. Volume is exactly 0
        if volume == 0:
            print(f"Problem 3 (Zero volume): {ticker} on {date}")

Problem 2 (String close): MSFT on 2026-09-29 (value: '498.75')
Problem 3 (Zero volume): MSFT on 2026-09-29
Problem 4 (Missing Field): KO company dictionary is missing 'sector'
Problem 1 (None close): KO on 2026-09-29


**TODO:** your answer to the last question: _..._


Genuinely missing data:

KO's None close on 2026-09-29 (the price value is literally missing/null).

KO's missing 'sector' key (the metadata attribute was omitted entirely).

Real measurements / Data type issues:

MSFT's string close ('498.75'): This is a valid numerical measurement that was mistakenly ingested as text/string rather than a float.

MSFT's zero volume (0): This is a valid physical measurement indicating no shares were traded on that date (e.g., market halt or zero volume reported), not a missing value.

### Step 5 — flatten to one row per ticker per day

`record_path` is the list you want one row from. `meta` is everything above it that should be copied down onto each of those rows. `sector` only exists on two of the three companies, so you will need `errors='ignore'` to keep `json_normalize` from refusing.

*Expected: 9 rows.*

In [6]:
# TODO: px = pd.json_normalize(holdings, record_path=..., meta=[...], sep='.',
#                              errors='ignore')
# TODO: assert len(px) == 9
# TODO: print px.dtypes and find the column that is the wrong type


# Flatten holdings down to price records
px = pd.json_normalize(
    data['portfolio']['holdings'],
    record_path='prices',
    meta=['ticker', 'shares', ['company', 'name'], ['company', 'sector']],
    sep='.',
    errors='ignore'
)

# Confirm length is 9
assert len(px) == 9, f"Expected 9 rows, got {len(px)}"

# Inspect column data types
print(px.dtypes)
print("\nFirst few rows:")
print(px.head())

date              object
close             object
volume             int64
ticker            object
shares            object
company.name      object
company.sector    object
dtype: object

First few rows:
         date   close    volume ticker shares     company.name company.sector
0  2026-09-28   231.4  52300000   AAPL    120       Apple Inc.     Technology
1  2026-09-29  229.85  48100000   AAPL    120       Apple Inc.     Technology
2  2026-09-30   233.1  55600000   AAPL    120       Apple Inc.     Technology
3  2026-09-28   502.1  22100000   MSFT     45  Microsoft Corp.     Technology
4  2026-09-29  498.75         0   MSFT     45  Microsoft Corp.     Technology


### Step 6 — fix the type drift

One string in a column of numbers is enough to make the whole column text, and a text column will not average. `pd.to_numeric(..., errors='coerce')` converts what it can and turns the rest into `NaN`.

**TODO:** convert `close` to a number, then confirm the dtype changed and count how many nulls you are left with. Say out loud where that null came from — it is not the string.

In [7]:
# TODO: px['close'] = pd.to_numeric(...)
# TODO: check the dtype and the null count

# Convert close column to numeric, forcing unparseable values (like None/null) to NaN
px['close'] = pd.to_numeric(px['close'], errors='coerce')

# Check the new data type
print("Updated close dtype:", px['close'].dtype)

# Count null values in the close column
null_count = px['close'].isna().sum()
print("Null count in close column:", null_count)

Updated close dtype: float64
Null count in close column: 1


### Step 7 — answer the questions

**TODO:** produce each of these, and report `count` next to every average so a reader can see how much data it rests on.

1. Mean closing price per ticker, and the number of days each mean is built from.
2. Each position's value on **2026-09-30** — that is `shares × close` — plus the total portfolio value and which holding is the largest.
3. Mean daily volume across all nine rows. Then compute it again with the zero-volume day excluded and report the difference.

That third one is the real question of the day. **Was dropping the zero the right call?** Defend your answer in a sentence.

In [8]:
# TODO: 1 -- mean close and day count per ticker
# TODO: 2 -- position values on the last date, total, and the largest
# TODO: 3 -- mean volume with and without the zero day

# 1. Mean closing price per ticker & number of valid days count
mean_close_summary = px.groupby('ticker')['close'].agg(
    mean_close='mean',
    day_count='count'
)
print("--- 1. Mean Close & Day Count Per Ticker ---")
print(mean_close_summary)
print("\n")

# 2. Position values on 2026-09-30 (shares * close), total portfolio value, & largest holding
px_last = px[px['date'] == '2026-09-30'].copy()
px_last['position_value'] = px_last['shares'] * px_last['close']

total_portfolio_value = px_last['position_value'].sum()
largest_holding = px_last.loc[px_last['position_value'].idxmax()]

print("--- 2. Positions on 2026-09-30 ---")
print(px_last[['ticker', 'shares', 'close', 'position_value']])
print(f"\nTotal Portfolio Value on 2026-09-30: ${total_portfolio_value:,.2f}")
print(f"Largest Holding: {largest_holding['ticker']} (${largest_holding['position_value']:,.2f})")
print("\n")

# 3. Mean daily volume with and without the zero-volume day
mean_vol_all = px['volume'].mean()
mean_vol_no_zero = px[px['volume'] > 0]['volume'].mean()
diff = mean_vol_no_zero - mean_vol_all

print("--- 3. Volume Analysis ---")
print(f"Mean daily volume (all 9 records): {mean_vol_all:,.2f}")
print(f"Mean daily volume (excluding 0 volume): {mean_vol_no_zero:,.2f}")
print(f"Difference: {diff:,.2f}")

--- 1. Mean Close & Day Count Per Ticker ---
        mean_close  day_count
ticker                       
AAPL       231.450          3
KO          71.075          2
MSFT       502.050          3


--- 2. Positions on 2026-09-30 ---
  ticker shares  close position_value
2   AAPL    120  233.1        27972.0
5   MSFT     45  505.3        22738.5
8     KO    300   70.9        21270.0

Total Portfolio Value on 2026-09-30: $71,980.50
Largest Holding: AAPL ($27,972.00)


--- 3. Volume Analysis ---
Mean daily volume (all 9 records): 26,677,777.78
Mean daily volume (excluding 0 volume): 30,012,500.00
Difference: 3,334,722.22


**TODO:** the zero-volume question: Yes, keeping the zero in would distort liquidity metrics. A reported volume of zero represents a non-trading event or market halt rather than standard trading activity; including it artificially depresses the true average daily trading volume of the liquid assets.

### Step 8 — write the result back out

Your output is somebody else's input, so send the context with it — the date and the currency, not a bare table of numbers.

**TODO:** write your per-ticker summary to `portfolio_summary.json` with `indent=2` and `allow_nan=False`, then print the file back. If `allow_nan=False` raises, that is the point of it — a `NaN` reached your output and you have to decide what it should say before it leaves your machine.

In [9]:
# TODO: build the result dict, json.dump it, and print the file back
import json

# 1. Prepare the dictionary, converting NaN to None (which serializes cleanly as null in JSON)
summary_dict = mean_close_summary.reset_index().to_dict(orient='records')
for row in summary_dict:
    if pd.isna(row['mean_close']):
        row['mean_close'] = None

output_payload = {
    "as_of": data['as_of'],
    "currency": data['currency'],
    "summary": summary_dict
}

# 2. Write to portfolio_summary.json with allow_nan=False
with open('portfolio_summary.json', 'w') as f:
    json.dump(output_payload, f, indent=2, allow_nan=False)

# 3. Read and print the file back to verify output
with open('portfolio_summary.json', 'r') as f:
    print(f.read())

{
  "as_of": "2026-09-30",
  "currency": "USD",
  "summary": [
    {
      "ticker": "AAPL",
      "mean_close": 231.45000000000002,
      "day_count": 3
    },
    {
      "ticker": "KO",
      "mean_close": 71.075,
      "day_count": 2
    },
    {
      "ticker": "MSFT",
      "mean_close": 502.05,
      "day_count": 3
    }
  ]
}


---

## Part 2 — a second payload, if you have time

This one is already formatted, and it is deeper: a game-day response nested three levels down. Nothing here is new — it is the same `json_normalize` from Step 5 reaching further down, then the `merge` you have been using since Sep 14. The only twist is that you pull **two** tables out of one payload and join them back together.

In [10]:
response = {
    'game': 'UVA vs Duke',
    'date': '2026-10-10',
    'attendance': 47200,
    'zones': [
        {'zone': 'A', 'capacity': 400, 'vendors': [
            {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
             'sales': {'orders': 300, 'revenue': 2250.0},
             'lines': [{'item': 'Cheeseburger', 'qty': 180},
                       {'item': 'Hot Dog', 'qty': 120}]},
            {'vendor_id': 'V-07', 'name': 'Stadium Dogs',
             'sales': {'orders': 210},
             'lines': [{'item': 'Hot Dog', 'qty': 210}]},
        ]},
        {'zone': 'B', 'capacity': 150, 'vendors': [
            {'vendor_id': 'V-12', 'name': 'Wahoo Wings',
             'sales': {'orders': 150, 'revenue': 1800.0},
             'lines': [{'item': 'Wings 6pc', 'qty': 90},
                       {'item': 'Wings 12pc', 'qty': 60}]},
        ]},
        {'zone': 'C', 'capacity': 120, 'vendors': [
            {'vendor_id': 'V-18', 'name': 'Rally Rain Gear',
             'sales': {'orders': 95, 'revenue': 570.0},
             'lines': [{'item': 'Rain Poncho', 'qty': 95}]},
        ]},
    ],
}
print('vendors:', sum(len(z['vendors']) for z in response['zones']))
print('line items:', sum(len(v['lines'])
                         for z in response['zones'] for v in z['vendors']))

vendors: 4
line items: 6


Four vendors, six line items.

**TODO A.** Flatten to one row per vendor, carrying down the game, the date, and the zone's `zone` and `capacity`. Use `sep='_'` and leave the column names it gives you alone. *4 rows.*

**TODO B.** One vendor is missing a field. Find it, and add a boolean column recording whether revenue was reported. Do **not** fill it with zero — write a comment saying why that would be wrong here.

**TODO C.** Flatten the same payload again, down to line items this time, keeping `vendor_id` and `zone`. *6 rows, quantities summing to 755.*

**TODO D.** Join the two, so each line item carries its vendor name and zone capacity. The vendor ID is not spelled the same way in both tables, so reach for `left_on` and `right_on` the way you did on Sep 16. Use `indicator=True` and confirm you still have 6 rows with nothing unmatched.

In [11]:
# TODO A, B, C, D
# --- TODO A: Flatten to vendors (1 row per vendor) ---
df_vendors = pd.json_normalize(
    response['zones'],
    record_path='vendors',
    meta=['zone', 'capacity'],
    meta_prefix='',
    record_prefix='',
    sep='_'
)
# Add top-level fields
df_vendors['game'] = response['game']
df_vendors['date'] = response['date']

print("--- TODO A: Vendors DataFrame (4 rows expected) ---")
print(f"Row count: {len(df_vendors)}")
print(df_vendors[['vendor_id', 'name', 'zone', 'capacity', 'sales_orders', 'sales_revenue']])
print("\n")


# --- TODO B: Check missing revenue field & add boolean indicator ---
# Finding missing field: Stadium Dogs (V-07) is missing 'sales_revenue'
df_vendors['has_revenue_reported'] = df_vendors['sales_revenue'].notna()

# Why filling with 0 is wrong:
# Filling missing revenue with 0 implies $0 in sales, whereas the vendor actually generated revenue from 210 orders;
# the revenue data was simply omitted/missing, not zero.

print("--- TODO B: Missing Revenue Check ---")
print(df_vendors[['vendor_id', 'name', 'sales_orders', 'sales_revenue', 'has_revenue_reported']])
print("\n")


# --- TODO C: Flatten to line items (6 rows expected, quantities sum to 755) ---
df_lines = pd.json_normalize(
    response['zones'],
    record_path=['vendors', 'lines'],
    meta=['zone', ['vendors', 'vendor_id']],
    sep='_'
)

print("--- TODO C: Line Items DataFrame (6 rows expected) ---")
print(f"Row count: {len(df_lines)}")
print(f"Total quantity sum: {df_lines['qty'].sum()} (Expected: 755)")
print(df_lines)
print("\n")


# --- TODO D: Merge vendors and line items ---
# The vendor ID column is 'vendor_id' in df_vendors and 'vendors_vendor_id' in df_lines
df_merged = pd.merge(
    df_lines,
    df_vendors,
    left_on=['zone', 'vendors_vendor_id'],
    right_on=['zone', 'vendor_id'],
    how='inner',
    indicator=True
)

print("--- TODO D: Merged DataFrame ---")
print(f"Merged row count: {len(df_merged)} (Expected: 6)")
print("Merge indicator counts:")
print(df_merged['_merge'].value_counts())
print("\nFinal Merged Table Preview:")
print(df_merged[['zone', 'vendor_id', 'name', 'capacity', 'item', 'qty', 'sales_orders', 'sales_revenue']])

--- TODO A: Vendors DataFrame (4 rows expected) ---
Row count: 4
  vendor_id             name zone capacity  sales_orders  sales_revenue
0      V-01     Hoos Burgers    A      400           300         2250.0
1      V-07     Stadium Dogs    A      400           210            NaN
2      V-12      Wahoo Wings    B      150           150         1800.0
3      V-18  Rally Rain Gear    C      120            95          570.0


--- TODO B: Missing Revenue Check ---
  vendor_id             name  sales_orders  sales_revenue  \
0      V-01     Hoos Burgers           300         2250.0   
1      V-07     Stadium Dogs           210            NaN   
2      V-12      Wahoo Wings           150         1800.0   
3      V-18  Rally Rain Gear            95          570.0   

   has_revenue_reported  
0                  True  
1                 False  
2                  True  
3                  True  


--- TODO C: Line Items DataFrame (6 rows expected) ---
Row count: 6
Total quantity sum: 755 (Expe

---

## Checkpoint (participation)

Report your row counts from the portfolio payload and what you decided about the zero-volume day.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# Checkpoint
holdings_count = 3       # TODO: should be 3
price_rows = 9           # TODO: should be 9
largest_position = 'AAPL'   # TODO: which ticker, on 2026-09-30
null_vs_zero = 'KO close is missing data (null), MSFT string close and MSFT zero volume are real measurements'       # TODO: which value was missing, which was a real measurement
zero_volume_call = 'Dropped the zero because it gives an accurate reflection of the average daily volume when the market and stock were actually open and trading'   # TODO: did you keep the zero in the mean, and why

print('holdings:', holdings_count)
print('price rows:', price_rows)
print('largest position:', largest_position)
print('null vs zero:', null_vs_zero)
print('zero volume:', zero_volume_call)

holdings: 3
price rows: 9
largest position: AAPL
null vs zero: KO close is missing data (null), MSFT string close and MSFT zero volume are real measurements
zero volume: Dropped the zero because it gives an accurate reflection of the average daily volume when the market and stock were actually open and trading
